# Header placeholder

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import resource
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

import apl

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()
D = apl.load()
recs, preps = D["recs"], D["preps"]
print(f"CRCNS data set {apl.DATASET}, version {apl.VERSION}; read {len(D['md5'])} files from {apl.BUCKET} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
print("MD5 of each file read, against the data set's checksums.md5:")
for name, h in D["md5"].items():
    print(f"  {name:24s} {h}  {'matches' if D['listed_md5'].get(name) == h else 'DIFFERS from ' + str(D['listed_md5'].get(name))}")
info = pd.DataFrame([{
    "prep": r["prep"], "bout": r["bout"], "file": r["file"], "Bout_field": r["Bout_field"],
    "neurons": int(np.unique(r["ids"]).size), "max_id": int(r["ids"].max()),
    "spikes": r["t"].size, "first_spike_s": r["t"].min().round(3), "last_spike_s": r["t"].max().round(2),
    "neurons_spiking_after_30s": int(np.unique(r["ids"][r["t"] >= 30]).size)} for r in recs])
print("\nRecordings per preparation:", info.groupby("prep").size().to_dict())
print(info.to_string(index=False))
pt = pd.DataFrame(preps)
pt["neurons_in_spikes"] = [int(info[info.prep == p].neurons.iloc[0]) for p in pt.prep]
pt["coord_points"] = [c["x"].size for c in D["coords"]]
pt["coord_file"] = [c["file"] for c in D["coords"]]
pt["coord_ganglion"] = [c["ganglion"] for c in D["coords"]]
print("\nFileTable.mat against the spike and coordinate files:")
print(pt.to_string(index=False))
print("\nNeurons per preparation: min", pt.n_neurons_table.min(), "max", pt.n_neurons_table.max(),
      "total", pt.n_neurons_table.sum(), "(description: 75 to 160, 1131 in total)")
print("Same neuron count in all three bouts of every preparation:",
      bool(info.groupby("prep").neurons.nunique().eq(1).all()))

The data set holds 30 recordings, three bouts in each of 10 preparations, in the order of
FileTable.mat, which the description says is the paper's numbering. Every neuron id from 1 to N is
present in every bout of its preparation, and every neuron spikes after 30 s. The spike files carry
no bout number (`Bout` is empty); the bout is read from the file name (da01, da02, da03). The
neuron counts of FileTable.mat match the spike files and the coordinate files. The MD5 sums printed
above are the version pinned (CRCNS data sets are not versioned).

In [ ]:
# Units, ranges, the clock of the stimulus, and recordings deposited twice.
rows = []
for r in recs:
    t = r["t"]
    pre = np.sum((t >= 25) & (t < 30)) / 5
    stim = np.sum((t >= 30) & (t < 32.5)) / 2.5
    post = np.sum((t >= 32.5) & (t < 37.5)) / 5
    rows.append({"file": r["file"], "pop_rate_25_30": round(pre, 1), "pop_rate_30_32.5": round(stim, 1),
                 "pop_rate_32.5_37.5": round(post, 1)})
clock = pd.DataFrame(rows)
print("Population spike rate (spikes/s) before, during and after the stimulus at 30 to 32.5 s:")
print(clock.to_string(index=False))
print("Recordings whose rate after 32.5 s exceeds that before 30 s:",
      int((clock["pop_rate_32.5_37.5"] > clock["pop_rate_25_30"]).sum()), "of", len(clock))
# P10 nerve recordings
rows = []
for q in D["p10"]:
    t = q["t"]
    rows.append({"file": q["file"], "prep": q["prep"], "bout": q["bout"], "spikes": t.size,
                 "ids": np.unique(q["ids"]).tolist(), "time_sorted": q["sorted"],
                 "range_s": f"{t.min():.2f} to {t.max():.2f}",
                 "min_interval_ms": round(np.diff(t).min() * 1000, 3),
                 "rate_25_30": round(np.sum((t >= 25) & (t < 30)) / 5, 1),
                 "rate_32.5_37.5": round(np.sum((t >= 32.5) & (t < 37.5)) / 5, 1)})
print("\nP10 recordings:")
print(pd.DataFrame(rows).to_string(index=False))
print("Smallest P10 interval as a sampling period: 1 /", round(1 / np.diff(D["p10"][0]["t"]).min(), 1), "Hz")
# duplicate recordings: identical spike trains between any two recordings
sig = {}
for r in recs:
    for i, tr in enumerate(apl.neuron_trains(r)):
        if tr.size > 5:
            sig.setdefault(tuple(np.round(tr, 4)), []).append((r["file"], i + 1))
dup = [v for v in sig.values() if len(v) > 1]
print("\nNeuron spike trains (more than 5 spikes) found identical in two places:", len(dup), dup[:5])
whole = {}
for r in recs:
    whole.setdefault((r["t"].size, round(float(r["t"].sum()), 3)), []).append(r["file"])
print("Recordings with identical spike counts and sums:", [v for v in whole.values() if len(v) > 1])
pd10 = {}
for q in D["p10"]:
    pd10.setdefault((q["t"].size, round(float(q["t"].sum()), 3)), []).append(q["file"])
print("P10 recordings deposited twice:", [v for v in pd10.values() if len(v) > 1])

In [ ]:
# Is the P10 nerve on the clock of the population? Cross-correlation of 1 s binned rates after the
# stimulus, against the same with the P10 series shifted by a random amount (circularly).
rng = np.random.default_rng(11)
rows = []
for q in D["p10"]:
    r = next(r for r in recs if r["prep"] == q["prep"] and r["bout"] == q["bout"])
    edges = np.arange(33, 122, 0.5)
    pop = np.histogram(r["t"], edges)[0].astype(float)
    nerve = np.histogram(q["t"], edges)[0].astype(float)
    lags = np.arange(-10, 11)
    cc = [np.corrcoef(np.roll(pop, l), nerve)[0, 1] for l in lags]
    shuffled = [np.corrcoef(np.roll(pop, int(rng.integers(20, len(pop) - 20))), nerve)[0, 1] for _ in range(200)]
    rows.append({"file": q["file"], "peak_lag_s": lags[int(np.nanargmax(np.abs(cc)))] * 0.5,
                 "r_at_lag_0": round(cc[10], 3), "max_abs_r": round(np.nanmax(np.abs(cc)), 3),
                 "shifted_95pct_abs_r": round(np.nanpercentile(np.abs(shuffled), 95), 3)})
print("Population rate against P10 rate, 0.5 s bins from 33 s, lags of -5 to 5 s:")
print(pd.DataFrame(rows).to_string(index=False))
# Are neuron ids the same neurons across bouts? Per-neuron rates after onset, correlated between
# bouts of a preparation, against the same with the ids permuted.
rows = []
for p in range(1, 11):
    rs = [np.array([np.sum(tr >= 30) for tr in apl.neuron_trains(r)]) for r in recs if r["prep"] == p]
    for a, b in apl.PAIRS:
        rho = stats.spearmanr(rs[a], rs[b])[0]
        null = [stats.spearmanr(rs[a], rng.permutation(rs[b]))[0] for _ in range(200)]
        rows.append({"prep": p, "pair": f"{a + 1},{b + 1}", "spearman": round(rho, 3),
                     "permuted_95pct": round(np.percentile(null, 95), 3)})
idt = pd.DataFrame(rows)
print("\nRank correlation of neurons' post-onset spike counts between bouts (same id):")
print(idt.pivot(index="prep", columns="pair", values="spearman").to_string())
print("Pairs above the 95th percentile of permuted ids:", int((idt.spearman > idt.permuted_95pct).sum()), "of", len(idt))

In [ ]:
# Coordinates: units and ranges, against the 501 x 501 diode template (plot_maps.m plots x*10, y*10)
for c, p in zip(D["coords"], preps):
    print(f"prep {p['prep']:2d} {c['file']:14s} {c['ganglion']:24s} n={c['x'].size:3d} x {np.nanmin(c['x']):5.1f} to {np.nanmax(c['x']):5.1f}  "
          f"y {np.nanmin(c['y']):5.1f} to {np.nanmax(c['y']):5.1f}  missing {int(np.isnan(c['x']).sum())}")
tpl = D["template"]
ys, xs = np.nonzero(tpl)
print("Diode template", tpl.shape, "nonzero x", xs.min(), "to", xs.max(), "y", ys.min(), "to", ys.max())

Data check notes placeholder

## dataset-size

> Three bouts of fictive locomotion were evoked in each of 10 isolated preparations (30 population
> responses); each recorded population held 120 to 180 neurons, 1131 neurons in all.

In [ ]:
n_preps = info.prep.nunique()
n_recs = len(info)
per_prep = info.groupby("prep").neurons.first()
print("preparations", n_preps, "| recordings", n_recs, "| neurons over preparations", int(per_prep.sum()))
print("neurons per recording: min", info.neurons.min(), "max", info.neurons.max())
print("recordings with 108 to 198 neurons:", int(info.neurons.between(108, 198).sum()), "of", n_recs)
print("variant, neurons spiking after onset: min", info.neurons_spiking_after_30s.min(), "max",
      info.neurons_spiking_after_30s.max(), "total over preparations",
      int(info.groupby("prep").neurons_spiking_after_30s.first().sum()))
print("per preparation:", per_prep.to_dict())

## fig1d-slow-oscillation

> After stimulation the spike-train power spectra of the populations are dominated by a slow
> oscillation at or below 0.1 Hz.

Chronux `mtspectrumpt` (ported in `apl.mtspectrumpt`, default tapers [3 5], 1 ms, fpass 0 to 1 Hz)
of each neuron's spikes after onset, averaged over the neurons of a recording.

In [ ]:
t0 = time.time()
spectra = []
for r in recs:
    trains = [tr[tr >= apl.STIM_ON] for tr in apl.neuron_trains(r)]
    f, S = apl.mtspectrumpt(trains)
    spectra.append(S)
spectra = np.array(spectra)
grand = spectra.mean(0)
fpk = np.array([apl.peak_frequency(f, S) for S in spectra])
fmax = np.array([f[1:][np.argmax(S[1:])] for S in spectra])
print(f"spectra in {time.time() - t0:.0f} s; frequency grid {f[0]:.4f} to {f[-1]:.4f} Hz, step {f[1] - f[0]:.4f} Hz")
print("grand mean spectrum: weighted peak", round(apl.peak_frequency(f, grand), 4), "Hz; maximum above 0 Hz at",
      round(f[1:][np.argmax(grand[1:])], 4), "Hz")
print("per recording, weighted peak (code): min", fpk.min().round(4), "median", np.median(fpk).round(4), "max", fpk.max().round(4))
print("recordings with weighted peak above 0 and at or below 0.11 Hz:", int(((fpk > 0) & (fpk <= 0.11)).sum()), "of 30")
print("variant, frequency of maximum power above 0 Hz: min", fmax.min().round(4), "median", np.median(fmax).round(4),
      "max", fmax.max().round(4), "; at or below 0.11 Hz:", int((fmax <= 0.11).sum()), "of 30")
print(pd.DataFrame({"file": [r["file"] for r in recs], "weighted_peak_Hz": fpk.round(4), "max_power_Hz": fmax.round(4)}).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3))
for S in spectra:
    ax.plot(f, S, color="0.7", lw=0.5)
ax.plot(f, grand, "k")
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("power"); ax.set_title("post-onset spectra (all points shown)")
plt.show()

## fig1e-decaying-rate

> The population firing rate decays slowly after the stimulus, decreasing in most evoked responses.

In [ ]:
rows = []
for r in recs:
    t = r["t"]
    starts = np.arange(30, r["t"].max() - 20 + 1e-9, 5)
    counts = np.array([np.sum((t >= s) & (t < s + 20)) for s in starts])
    R = np.corrcoef(starts + 10, counts)[0, 1]
    edges = np.arange(32.5, r["t"].max(), 1.0)
    rate = np.histogram(t, edges)[0]
    slope = np.polyfit(edges[:-1] + 0.5, rate, 1)[0]
    rows.append({"prep": r["prep"], "bout": r["bout"], "R_20s_windows": R, "n_windows": len(starts), "slope_spikes_per_s2": slope})
rate_tab = pd.DataFrame(rows)
print(rate_tab.round(3).to_string(index=False))
print("(a) decreasing, R < -0.2:", int((rate_tab.R_20s_windows < -0.2).sum()), "of 30; increasing, R > 0.2:",
      int((rate_tab.R_20s_windows > 0.2).sum()))
print("(b) negative slope from 32.5 s:", int((rate_tab.slope_spikes_per_s2 < 0).sum()), "of 30")
print("per preparation, (a) decreasing:", rate_tab.assign(dec=rate_tab.R_20s_windows < -0.2).groupby("prep").dec.sum().to_dict())

## fig2b-dimensions

> Capturing 80% of the population variance needs 4 to 8 embedding dimensions in the evoked
> responses, a reduction by more than a factor of 10 from the number of neurons.

The Gaussian SD is read four ways (helper docstring) and the count two ways, as the criterion
lists them.

In [ ]:
sds = apl.gaussian_sds(recs)
print("Gaussian SD (s) by reading: ", {k: (round(v.min(), 3), round(v.max(), 3)) for k, v in sds.items()})
dims = {}
t0 = time.time()
for mode in ["sqrt12", "twelve", "own", "pooled"]:
    evs = [apl.Program(r, s).ev for r, s in zip(recs, sds[mode])]
    for rule in ["min80", "le80"]:
        dims[(mode, rule)] = np.array([apl.n_pcs(ev, 0.8, rule) for ev in evs])
print(f"PCA under every reading in {time.time() - t0:.0f} s")
nn = info.neurons.values
rows = []
for (mode, rule), d in dims.items():
    rows.append({"sd": mode, "count": rule, "min": d.min(), "max": d.max(), "in_3.6_to_8.8": int(((d >= 3.6) & (d <= 8.8)).sum()),
                 "min_neurons_over_dims": round((nn / d).min(), 1), "ratio_over_10": int((nn / d > 10).sum())})
dim_tab = pd.DataFrame(rows)
print(dim_tab.to_string(index=False))
print("\ncounts per recording (min80 | le80) for the code's width (own median ISI):")
print(pd.DataFrame({"file": info.file, "own_min80": dims[("own", "min80")], "own_le80": dims[("own", "le80")],
                    "pooled_min80": dims[("pooled", "min80")], "pooled_le80": dims[("pooled", "le80")], "neurons": nn}).to_string(index=False))

## The recurrence analysis

Run once per reading of the width and the count. The local linear models (claims fig4 to fig5)
are fitted for the four readings with the code's width; the two narrow widths give 22 to 75
dimensions (claim fig2b-dimensions) and are carried only through the dominant period
(fig2ef-periodic-recurrence). The adopted reading, used where one reading has to be chosen, is
`own/min80`: the median ISI of the recording (the width the code uses; the Methods compute it
"in population", the recording) and the smallest number of PCs reaching 80% (the Methods, "at least
80% of the variance"). The other readings are shown with it.

In [ ]:
t0 = time.time()
RUNS = {}
for mode in ["own", "pooled"]:
    for rule in ["min80", "le80"]:
        RUNS[f"{mode}/{rule}"] = apl.run_all(recs, sds[mode], rule, lld=True, workers=4)
for mode in ["sqrt12", "twelve"]:
    for rule in ["min80", "le80"]:
        RUNS[f"{mode}/{rule}"] = apl.run_all(recs, sds[mode], rule, lld=False, workers=4)
SUM = {k: pd.DataFrame([apl.summarise(o) for o in v]) for k, v in RUNS.items()}
print(f"recurrence analysis of 30 recordings under {len(RUNS)} readings in {time.time() - t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB per worker")
ADOPT = "own/min80"
print("checked points per recording:", SUM[ADOPT].n_points.min(), "to", SUM[ADOPT].n_points.max(), "(Methods: around 7770)")

## fig2ef-periodic-recurrence

> The evoked low-dimensional activity recurs with a regular period; strongly periodic recurrence on
> the scale of 10 to 15 s is found in many but not all of the 30 responses.

In [ ]:
rows = []
for k, S in SUM.items():
    for stat in ["period_mean", "period_median"]:
        v = S[stat].values
        rows.append({"reading": k, "period": stat, "n_with_peak": int(np.isfinite(v).sum()),
                     "in_9_to_16.5": int(((v >= 9) & (v <= 16.5)).sum()), "min": np.nanmin(v).round(2),
                     "median": np.nanmedian(v).round(2), "max": np.nanmax(v).round(2)})
per_tab = pd.DataFrame(rows)
per_tab["reproduced"] = (per_tab["in_9_to_16.5"] > 15) & (per_tab["in_9_to_16.5"] < 30)
print(per_tab.to_string(index=False))
S = SUM[ADOPT]
print("\nadopted reading, per recording:")
print(S[["prep", "bout", "d", "n_peaks", "prop_recurrent", "period_mean", "period_median"]].round(3).to_string(index=False))
print("by preparation, mean period:", S.groupby("prep").period_mean.mean().round(2).to_dict())
print("adopted reading, quartiles of the dominant period (mean of the peak):", np.percentile(S.period_mean, [25, 75]).round(2),
      "; outside 9 to 16.5 s:", S[~S.period_mean.between(9, 16.5)][["prep", "bout", "period_mean"]].round(2).values.tolist())

## fig2e-single-dominant-orbit

> Almost all populations are dominated by a single periodic orbit; the majority of programs have
> more than 50% of their recurrent points in the dominant peak.

In [ ]:
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    S = SUM[k]
    rows.append({"reading": k, "over_50pct_hist": int((S.prop_dom_hist > 0.5).sum()),
                 "over_50pct_all_recurrent": int((S.prop_dom_all > 0.5).sum()),
                 "peaks_min": S.n_peaks.min(), "peaks_max": S.n_peaks.max(),
                 "peaks_counts": S.n_peaks.value_counts().sort_index().to_dict()})
print(pd.DataFrame(rows).to_string(index=False))
print("\nadopted reading:")
print(SUM[ADOPT][["prep", "bout", "n_peaks", "dom_n", "prop_dom_hist", "prop_dom_all"]].round(3).to_string(index=False))

## fig2s2-threshold-robust

> The periodic orbits are robust to the threshold for recurrence, with similar periodicity at
> thresholds of 10%, 5% and 2%.

In [ ]:
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    P = {th: pd.DataFrame([apl.summarise(o, i) for o in RUNS[k]]) for i, th in enumerate(apl.THETA_PRCT)}
    for stat in ["period_mean", "period_median"]:
        p10 = P[10][stat].values
        for th in (5, 2):
            pl = P[th][stat].values
            ok = np.isfinite(p10) & np.isfinite(pl)
            rho, p = stats.spearmanr(p10[ok], pl[ok])
            mad = np.median(np.abs(pl[ok] - p10[ok]))
            rows.append({"reading": k, "period": stat, "theta": f"{th}%", "n_both": int(ok.sum()),
                         "spearman": round(rho, 3), "p": p, "(a)": bool(rho > 0 and p < 0.05),
                         "median_abs_diff_s": round(mad, 2), "10pct_of_median": round(0.1 * np.median(p10[ok]), 2),
                         "(b)": bool(mad < 0.1 * np.median(p10[ok]))})
thr_tab = pd.DataFrame(rows)
print(thr_tab.to_string(index=False))
P = {th: pd.DataFrame([apl.summarise(o, i) for o in RUNS[ADOPT]]) for i, th in enumerate(apl.THETA_PRCT)}
print("\nadopted reading, mean dominant period at each threshold:")
print(pd.DataFrame({th: P[th].period_mean.round(2) for th in P}).assign(file=info.file).to_string(index=False))

## fig3a-coalescence

> Almost all evoked responses reach a state of high recurrence within one oscillation period of
> stimulation and are thereafter dominated by recurrence.

The coalescence time is the mid-point of the first 5 s window with more than 90% recurrent points.
Windows start at 35 s, where the checked points start (Methods), so the earliest possible value is
37.5 s, 7.5 s after onset.

In [ ]:
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    S = SUM[k]
    c_on = S.coalesce_t - apl.STIM_ON
    for stat in ["period_mean", "period_median"]:
        rows.append({"reading": k, "period": stat, "within_period_from_onset": int((c_on <= S[stat]).sum()),
                     "within_period_from_offset": int((c_on - 2.5 <= S[stat]).sum()),
                     "recurrent_after_over_50pct": int((S.prop_recurrent_after > 0.5).sum()),
                     "never_coalesced": int(S.coalesce_t.isna().sum())})
print(pd.DataFrame(rows).to_string(index=False))
S = SUM[ADOPT].assign(coal_from_onset=lambda d: d.coalesce_t - 30)
print("\nadopted reading: coalescence from onset, s: min", S.coal_from_onset.min(), "median", S.coal_from_onset.median(),
      "max", S.coal_from_onset.max(), "; proportion recurrent after it: min", S.prop_recurrent_after.min().round(3),
      "median", S.prop_recurrent_after.median().round(3))
print(S[["prep", "bout", "coal_from_onset", "period_mean", "period_median", "prop_recurrent_after"]].round(3).to_string(index=False))

## Comparisons between the programs of a preparation

Hausdorff distances on the axes of program 1 (claim fig3c) and on common axes (fig3s1a), with 100
neuron shuffles each way as control; the distances of Figure 3D; the similarity matrices of
Figure 3F. Run for the four readings with the code's width (the point sets come from each
reading's recurrence analysis).

In [ ]:
t0 = time.time()
MAN = {k: apl.run_manifold(recs, sds[k.split("/")[0]], RUNS[k], n_shuffle=100, seed=1)
       for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]}
print(f"comparisons in {time.time() - t0:.0f} s")
def haus_table(k, axes, ps):
    rows = []
    for m in MAN[k]:
        for (i, j) in apl.PAIRS:
            h, c = m[(axes, ps, i, j)]
            rows.append({"prep": m["prep"], "pair": f"{i + 1},{j + 1}", "H": h, "ctrl_mean": np.mean(c),
                         "ctrl_2sem": 2 * np.std(c, ddof=1) / np.sqrt(c.size), "dims": m["k_prog1" if axes == "prog1" else "k_common"]})
    return pd.DataFrame(rows)

## fig3c-same-manifold

> Sequentially evoked responses of a preparation fall on the same manifold; their distance is below
> that of random projections for all pairs except two, pairs (1,2) and (1,3) of preparation 4.

In [ ]:
rows = []
for k in MAN:
    for ps in ["recurrent", "code"]:
        T = haus_table(k, "prog1", ps)
        below = T.H < T.ctrl_mean
        rows.append({"reading": k, "points": ps, "pairs_below_control": int(below.sum()),
                     "failing_pairs": [(p, q) for p, q in zip(T.prep[~below], T.pair[~below])]})
print(pd.DataFrame(rows).to_string(index=False))
T = haus_table(ADOPT, "prog1", "recurrent")
print("\nadopted reading, recurrent points:")
print(T.assign(ratio=T.H / T.ctrl_mean).round(3).to_string(index=False))
T = haus_table(ADOPT, "prog1", "code")
print("\nadopted reading, the code's point sets:")
print(T.assign(ratio=T.H / T.ctrl_mean).round(3).to_string(index=False))

## fig3s1a-common-axes

> The convergence of sequential responses to the same manifold holds when the three programs are
> projected onto common axes from a PCA of their concatenated activity.

In [ ]:
rows = []
for k in MAN:
    for ps in ["recurrent", "code"]:
        T = haus_table(k, "common", ps)
        below = T.H < T.ctrl_mean
        rows.append({"reading": k, "points": ps, "pairs_below_control": int(below.sum()),
                     "failing_pairs": [(p, q) for p, q in zip(T.prep[~below], T.pair[~below])],
                     "dims": sorted(set(T.dims))})
print(pd.DataFrame(rows).to_string(index=False))
T = haus_table(ADOPT, "common", "code")
print("\nadopted reading, the code's point sets:")
print(T.assign(ratio=T.H / T.ctrl_mean).round(3).to_string(index=False))

## fig3d-initial-conditions

> Stimulation drives responses of the same preparation far apart before coalescence, more than at
> stimulus onset or after both have coalesced, so the same manifold is reached from different
> initial conditions.

The transient distance is the largest distance between a point of one program and a point of the
other between onset and their coalescence; for speed every fifth point of the second program is
used (a standard choice: neighbouring points 10 ms apart are nearly equal).

In [ ]:
rows = []
for k in MAN:
    for ps in ["code", "recurrent"]:
        for m in MAN[k]:
            for (i, j) in apl.PAIRS:
                on, tr = m[("ic", i, j)]
                rows.append({"reading": k, "points": ps, "prep": m["prep"], "pair": f"{i + 1},{j + 1}",
                             "onset": on, "transient": tr, "coalesced": m[("common", ps, i, j)][0]})
IC = pd.DataFrame(rows)
summ = IC.groupby(["reading", "points"]).apply(lambda d: pd.Series({
    "transient_over_both": int(((d.transient > d.onset) & (d.transient > d.coalesced)).sum()),
    "median_onset": d.onset.median(), "median_transient": d.transient.median(), "median_coalesced": d.coalesced.median()}))
print(summ.round(2).to_string())
print("\nadopted reading, code's point sets:")
print(IC[(IC.reading == ADOPT) & (IC.points == "code")].drop(columns=["reading", "points"]).round(2).to_string(index=False))

## fig3f-pairwise-similarity

> Pairwise similarity between neurons is preserved across responses of the same preparation, above a
> null model, except for pairs (1,3) and (2,3) of preparation 7.

In [ ]:
rows, detail = [], []
for k in MAN:
    for meth in ["spearman", "pearson"]:
        corr = (lambda a, b: stats.spearmanr(a, b)[0]) if meth == "spearman" else (lambda a, b: np.corrcoef(a, b)[0, 1])
        for ctl in ["first", "larger"]:
            n, fails = 0, []
            for m in MAN[k]:
                Sm, Em = m["sim"]
                cs = [corr(Sm[i].ravel(), Em[i].ravel()) for i in range(3)]
                for (i, j) in apl.PAIRS:
                    dat = corr(Sm[i].ravel(), Sm[j].ravel())
                    c = cs[i] if ctl == "first" else max(cs[i], cs[j])
                    if dat > c:
                        n += 1
                    else:
                        fails.append((m["prep"], f"{i + 1},{j + 1}"))
                    if k == ADOPT and ctl == "first":
                        detail.append({"corr": meth, "prep": m["prep"], "pair": f"{i + 1},{j + 1}", "data": dat,
                                       "control_first": cs[i], "control_second": cs[j]})
            rows.append({"reading": k, "corr": meth, "control": ctl, "pairs_above": n, "failing": fails})
print(pd.DataFrame(rows).to_string(index=False))
print("\nadopted reading:")
print(pd.DataFrame(detail).round(3).to_string(index=False))

## fig3h-perturbations-return

> Of 19 spontaneous divergences from the trajectory, 17 (90%) returned to a recurrent state; the two
> that did not were consistent with the end of the evoked locomotion.

A divergent period (Methods) is a run of windows after coalescence with recurrence density below
90% that falls below 50%; "returned" means a later window is above 90%. Periods whose divergent
point lies within two dominant periods of the end are removed (Methods), with the dominant period
read as the mean or the median of the peak. The code removes only periods that never return
(`Are_jumps_perturbations_or_transitions2.m`); that is shown as a variant.

In [ ]:
DIV = {}
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    for stat in ["period_mean", "period_median"]:
        allv = []
        for o, s in zip(RUNS[k], SUM[k].itertuples()):
            for d in apl.divergences(o, getattr(s, stat)):
                d.update(prep=o["prep"], bout=o["bout"])
                allv.append(d)
        V = pd.DataFrame(allv)
        DIV[(k, stat)] = V
        kept = V[~V.near_end]
        code_kept = V[V.returned | ~V.near_end]
        rows.append({"reading": k, "period": stat, "all_periods": len(V), "kept": len(kept),
                     "returned": int(kept.returned.sum()), "fraction": round(kept.returned.mean(), 3),
                     "code_rule_kept": len(code_kept), "code_rule_returned": int(code_kept.returned.sum())})
print(pd.DataFrame(rows).to_string(index=False))
V = DIV[(ADOPT, "period_mean")]
print("\nadopted reading, mean period:")
print(V[["prep", "bout", "t_point", "min_dens", "returned", "near_end"]].round(3).to_string(index=False))

## fig3i-same-manifold-after

> Of the 17 divergences that returned to a recurrent state, all but three returned to the same
> manifold; the three appeared to move to a different periodic attractor.

In [ ]:
rows = []
for (k, stat), V in DIV.items():
    kept = V[~V.near_end & V.returned]
    rows.append({"reading": k, "period": stat, "returned": len(kept),
                 "same_(a)_after_end": int((kept.p_after_end > 0.2).sum()),
                 "same_(b)_after_point": int((kept.p_after_point > 0.2).sum())})
print(pd.DataFrame(rows).to_string(index=False))
V = DIV[(ADOPT, "period_mean")]
print("\nadopted reading, returned periods:")
print(V[~V.near_end & V.returned][["prep", "bout", "t_point", "t_return", "n_before", "p_after_end", "p_after_point"]].round(3).to_string(index=False))

## fig4bc-stable-spiral

> The evoked responses have highly clustered maximum eigenvalues of the local linear dynamics,
> dominated by negative complex values, so the pedal ganglion implements a contracting periodic
> orbit, a stable spiral attractor, found in every preparation.

In [ ]:
EIG = {}
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    E = pd.DataFrame([{**{x: y for x, y in apl.emax_stats(o["eigs"], o["npts"]).items() if x != "eigs"},
                       "prep": o["prep"], "bout": o["bout"]} for o in RUNS[k]])
    E["spiral"] = (E.a < -0.0001) & (E.b > 0.0005)
    EIG[k] = E
    per_prep = E.groupby("prep").spiral.sum()
    rows.append({"reading": k, "mean_a": E.a.mean(), "mean_b": E.b.mean(), "sd_a": E.a.std(), "sd_b": E.b.std(),
                 "grand_mean_spiral": bool(E.a.mean() < -0.0001 and E.b.mean() > 0.0005),
                 "responses_spiral": int(E.spiral.sum()), "preps_2of3": int((per_prep >= 2).sum()),
                 "used_points_min": E.n_used.min(), "dropped_points": int((E.n_all - E.n_used).sum())})
print(pd.DataFrame(rows).to_string(index=False))
E = EIG[ADOPT]
print("\nadopted reading:")
print(E[["prep", "bout", "a", "b", "sd_a", "sd_b", "n_used", "n_all", "spiral"]].to_string(index=False))
print("spiral responses per preparation:", E.groupby("prep").spiral.sum().to_dict())
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.scatter(E.b, E.a, c=E.prep, cmap="tab10", s=15)
ax.axhline(-0.0001, color="0.6", lw=0.5); ax.axvline(0.0005, color="0.6", lw=0.5)
ax.set_xlabel("mean imaginary part b"); ax.set_ylabel("mean real part a")
ax.set_title(f"all 30 shown; a {E.a.min():.4f} to {E.a.max():.4f}")
plt.show()

## fig4c-orbital-period

> The linear model estimates a mean orbital period of about 10 s, consistent with the period from
> the recurrent points.

In [ ]:
rows = []
for k, E in EIG.items():
    lin = 2 * np.pi / E.b * 0.01
    S = SUM[k]
    r_mean = stats.pearsonr(lin, S.period_mean)
    r_med = stats.pearsonr(lin, S.period_median)
    rows.append({"reading": k, "mean_linear_period_s": lin.mean(), "variant_mean_of_mRot_s": E.mrot.mean(),
                 "pearson_r_vs_mean_recurrence": r_mean[0], "p": r_mean[1], "pearson_r_vs_median_recurrence": r_med[0], "p_med": r_med[1],
                 "min": lin.min(), "max": lin.max()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

## fig4s1a-contracting-dimensions

> In most responses the trajectory contracts in all embedding dimensions; a few responses have a
> positive real eigenvalue in one low-variance dimension, a minor expansion.

In [ ]:
rows, detail = [], []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    allneg = one_pos_later = 0
    for o in RUNS[k]:
        st = apl.emax_stats(o["eigs"], o["npts"])
        ed = apl.eig_dimensions(st["eigs"])
        mu = ed["mean_unique"]
        allneg += bool(np.all(mu < 0))
        one_pos_later += bool(np.sum(mu > 0) == 1 and mu[0] < 0)
        if k == ADOPT:
            detail.append({"prep": o["prep"], "bout": o["bout"], "unique_dims": mu.size, "positive": int(np.sum(mu > 0)),
                           "first_positive": bool(mu[0] > 0), "means": np.round(mu * 1000, 3).tolist()})
    rows.append({"reading": k, "all_negative": allneg, "one_positive_not_first": one_pos_later})
print(pd.DataFrame(rows).to_string(index=False))
print("\nadopted reading (means x 1000):")
print(pd.DataFrame(detail).to_string(index=False))

## fig4s1b-rotation-dominant

> Local trajectories were dominated by rotation throughout; in every preparation most local models
> have a complex eigenvalue among their top two.

In [ ]:
rows, detail = [], []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    pc = []
    for o in RUNS[k]:
        st = apl.emax_stats(o["eigs"], o["npts"])
        ed = apl.eig_dimensions(st["eigs"])
        pc.append({"prep": o["prep"], "bout": o["bout"], "p_complex12": ed["p_complex12"], "p_complex1": ed["p_complex1"]})
    pc = pd.DataFrame(pc)
    maj = pc.assign(o=pc.p_complex12 > 0.5).groupby("prep").o.sum()
    rows.append({"reading": k, "responses_over_0.5": int((pc.p_complex12 > 0.5).sum()), "min": pc.p_complex12.min(),
                 "preps_with_majority": int((maj >= 2).sum())})
    if k == ADOPT:
        detail = pc
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(detail.round(4).to_string(index=False))

## fig5b-period-changes

> Over a response the orbital period may slow, speed up or not change; all three occur among the 30
> responses.

In [ ]:
t0 = time.time()
PC = {}
rows = []
for k in ["own/min80", "pooled/le80", "own/le80", "pooled/min80"]:
    out = []
    for n, o in enumerate(RUNS[k]):
        w = o["windows"][-1]
        hi = np.flatnonzero(w["dens"] > 0.9)
        sl = slice(hi[0], hi[-1] + 1)
        wt = 1 / w["sddelay"]
        wt[~np.isfinite(wt)] = 0
        wt = wt * w["dens"]
        r, p, nw = apl.weighted_spearman(w["mid_t"][sl], w["mdelay"][sl], wt[sl], 10000, np.random.default_rng(500 + n))
        row = {"prep": o["prep"], "bout": o["bout"], "rho": r, "p": p, "n_windows": nw}
        if k == ADOPT:
            rc, pcode, _ = apl.weighted_spearman(w["mid_t"][sl], w["mdelay"][sl], wt[sl], 100000, np.random.default_rng(900 + n), pooled=True)
            row.update(p_code=pcode)
        out.append(row)
    X = pd.DataFrame(out)
    PC[k] = X
    rows.append({"reading": k, "slowing_p<0.01": int(((X.rho > 0) & (X.p < 0.01)).sum()),
                 "speeding_p<0.01": int(((X.rho < 0) & (X.p < 0.01)).sum()), "no_change": int((X.p >= 0.01).sum()),
                 "windows_min": X.n_windows.min(), "windows_max": X.n_windows.max()})
print(f"permutation tests in {time.time() - t0:.0f} s")
print(pd.DataFrame(rows).to_string(index=False))
X = PC[ADOPT]
print("\nadopted reading; p from 10000 permutations of the delays (two-sided), p_code from the code's 100000 pooled permutations (one-sided):")
print(X.round(4).to_string(index=False))
print("code test: slowing", int(((X.rho > 0) & (X.p_code < 0.01)).sum()), "speeding", int(((X.rho < 0) & (X.p_code < 0.01)).sum()),
      "no change", int((X.p_code >= 0.01).sum()))

## fig5c-no-relation-contraction

> There is no relationship between the change in orbital period over a response and its rate of
> contraction, as a stable spiral predicts.

In [ ]:
rows = []
for k, X in PC.items():
    E = EIG[k]
    r, p = stats.pearsonr(X.rho, E.a)
    contraction = np.exp(100 * E.a) * 100 - 100
    r2, p2 = stats.pearsonr(X.rho, contraction)
    rows.append({"reading": k, "R_rho_vs_a": r, "p": p, "R_rho_vs_contraction_pct": r2, "p_contraction": p2})
print(pd.DataFrame(rows).round(4).to_string(index=False))

## fig6s1-no-p10-motorneurons

> No recorded neuron is a P10 motorneuron; no neuron comes close to one-to-one locking of its spikes
> with P10 spikes at any delay, nor locks consistently across programs.

In [ ]:
rows = []
for q in D["p10"]:
    r = next(r for r in recs if r["prep"] == q["prep"] and r["bout"] == q["bout"])
    for i, tr in enumerate(apl.neuron_trains(r)):
        v, n = apl.p10_locking(tr, q["t"])
        rows.append({"prep": q["prep"], "bout": q["bout"], "neuron": i + 1, "maxP": v, "spikes_after_33s": n})
MN = pd.DataFrame(rows)
print("neuron-program values:", len(MN), "; without spikes after 33 s:", int(MN.maxP.isna().sum()))
print("largest maximum locking:", round(MN.maxP.max(), 3), "; at or above 0.9:", int((MN.maxP >= 0.9).sum()))
two = MN[MN.maxP >= 0.9].groupby(["prep", "neuron"]).size()
print("neurons at or above 0.9 in two programs:", int((two >= 2).sum()))
print("largest values:")
print(MN.sort_values("maxP", ascending=False).head(8).round(3).to_string(index=False))
for c in (0.5, 0.7):
    print(f"neuron-program values at or above {c}:", int((MN.maxP >= c).sum()))
print("per preparation, max:", MN.groupby("prep").maxP.max().round(3).to_dict())

## fig6d-decode-p10

> P10 nerve activity can be fitted and forecast from the low-dimensional trajectory in all nine
> responses with P10 recordings, so the trajectory encodes muscle commands.

Adopted width (median ISI of the recording); PCs for 80% (smallest number reaching it). The
Methods' windowing (fit 40 s, forecast 10 s, step 1 s, from 33 s; histories 50 to 200 ms) is the
main reading; the code's (training on 50% of the series, 0.2 s steps, histories 50 to 250 ms) is a
variant. The 90% models of claim fig6s2 use the Methods' windowing with histories of 50, 100, 150 and
200 ms only, to fit the time limit.

In [ ]:
t0 = time.time()
def p10_jobs(frac, windowing, hists):
    J = []
    for q in D["p10"]:
        i = next(i for i, r in enumerate(recs) if r["prep"] == q["prep"] and r["bout"] == q["bout"])
        J.append((recs[i], q["t"], float(sds["own"][i]), frac, windowing, hists))
    return J
DEC = {("80", "methods"): apl.run_decoding(p10_jobs(0.8, "methods", apl.HISTORIES_METHODS)),
       ("80", "code"): apl.run_decoding(p10_jobs(0.8, "code", apl.HISTORIES_CODE)),
       ("90", "methods"): apl.run_decoding(p10_jobs(0.9, "methods", [5, 10, 15, 20]))}
print(f"decoding in {time.time() - t0:.0f} s")
def dec_table(res, hists=None):
    rows = []
    for o in res:
        if hists is None:
            R = np.array([w["R"] for w in o["windows"]]); MAE = np.array([w["MAE"] for w in o["windows"]]); best = o["best"]
        else:
            best = min(hists, key=lambda h: np.nanmean(o["by_history"][h][1]))
            R, MAE = o["by_history"][best]
        ok = np.isfinite(R)
        sem = np.nanstd(R, ddof=1) / np.sqrt(ok.sum())
        rows.append({"prep": o["prep"], "bout": o["bout"], "dims": o["d"], "history_ms": best * 10, "windows": len(R),
                     "undefined_R": int((~ok).sum()), "mean_R": np.nanmean(R), "mean_R_minus_2sem": np.nanmean(R) - 2 * sem,
                     "mean_R2": np.nanmean(R ** 2), "mean_MAE": np.nanmean(MAE),
                     "train_R": np.nanmean([w["R_train"] for w in o["windows"]]) if hists is None else np.nan})
    return pd.DataFrame(rows)
for name, res, h in [("Methods windowing, 80%", DEC[("80", "methods")], None),
                     ("variant: histories up to 100 ms", DEC[("80", "methods")], list(range(5, 11))),
                     ("variant: code windowing", DEC[("80", "code")], None)]:
    T = dec_table(res, h)
    print(f"\n{name}: responses with mean R - 2 s.e.m. > 0: {int((T.mean_R_minus_2sem > 0).sum())} of 9; windows in all {T.windows.sum()}")
    print(T.round(3).to_string(index=False))

## fig6s2-dimensions-p10

> The nine recordings with P10 activity need only 3 to 5 embedding dimensions for 80% variance, and
> at least double that for 90%; using the 90% dimensions does not improve the P10 forecasts.

In [ ]:
p10_idx = [next(i for i, r in enumerate(recs) if r["prep"] == q["prep"] and r["bout"] == q["bout"]) for q in D["p10"]]
rows = []
for mode in ["sqrt12", "twelve", "own", "pooled"]:
    evs = [apl.Program(recs[i], sds[mode][i]).ev for i in p10_idx]
    for rule in ["min80", "le80"]:
        d80 = np.array([apl.n_pcs(ev, 0.8, rule) for ev in evs])
        d90 = np.array([apl.n_pcs(ev, 0.9, rule) for ev in evs])
        rows.append({"sd": mode, "count": rule, "d80": d80.tolist(), "d90": d90.tolist(),
                     "all_in_2.7_to_5.5": bool(np.all((d80 >= 2.7) & (d80 <= 5.5))), "d90_at_least_2x": bool(np.all(d90 >= 2 * d80))})
print(pd.DataFrame(rows).to_string(index=False))
T80 = dec_table(DEC[("80", "methods")]); T90 = dec_table(DEC[("90", "methods")])
cmp = pd.DataFrame({"prep": T80.prep, "bout": T80.bout, "dims80": T80.dims, "dims90": T90.dims,
                    "MAE80": T80.mean_MAE, "MAE90": T90.mean_MAE, "R2_80": T80.mean_R2, "R2_90": T90.mean_R2})
print(cmp.round(3).to_string(index=False))
w_mae = stats.wilcoxon(cmp.MAE90, cmp.MAE80)
w_r2 = stats.wilcoxon(cmp.R2_90, cmp.R2_80)
print("Wilcoxon signed-rank, MAE 90% against 80%: p =", round(w_mae.pvalue, 4), "; 90% lower in", int((cmp.MAE90 < cmp.MAE80).sum()), "of 9;",
      "median difference", round(float(np.median(cmp.MAE90 - cmp.MAE80)), 3))
print("Wilcoxon signed-rank, R2 90% against 80%: p =", round(w_r2.pvalue, 4), "; 90% higher in", int((cmp.R2_90 > cmp.R2_80).sum()), "of 9;",
      "median difference", round(float(np.median(cmp.R2_90 - cmp.R2_80)), 3))

## Participation

Per program, from its own PCA under the adopted width; d is (a) the program's own count for 80%
(Methods) or (b) the count from the PCA of the three programs concatenated, at most 80%, used for
all three (code). Normalised to the program's maximum.

In [ ]:
progs = [apl.Program(r, s, "min80") for r, s in zip(recs, sds["own"])]
PART = {}
for reading in ["a_own", "b_common"]:
    for p in range(1, 11):
        ix = sorted([i for i, r in enumerate(recs) if r["prep"] == p], key=lambda i: recs[i]["bout"])
        if reading == "b_common":
            _, _, evc = apl.pca(np.vstack([progs[i].sdf[progs[i].ix_stim] for i in ix]))
            ds = [apl.n_pcs(evc, 0.8, "le80")] * 3
        else:
            ds = [apl.n_pcs(progs[i].ev, 0.8, "min80") for i in ix]
        PART[(reading, p)] = (np.array([apl.participation(progs[i].coeffs, progs[i].ev, d)[1] for i, d in zip(ix, ds)]), ds)
print({k: v[1] for k, v in PART.items()})

## fig7a-long-tailed-participation

> Every response has a long-tailed distribution of single-neuron participation, a minority of
> neurons dominating its dynamics.

In [ ]:
rows = []
for (reading, p), (P, ds) in PART.items():
    for b in range(3):
        rows.append({"reading": reading, "prep": p, "bout": b + 1, "skewness": stats.skew(P[b]), "frac_over_half": np.mean(P[b] > 0.5),
                     "median": np.median(P[b])})
PA = pd.DataFrame(rows)
print(PA.groupby("reading").apply(lambda d: pd.Series({"skew_positive": int((d.skewness > 0).sum()), "skew_min": d.skewness.min(),
      "frac_over_half_below_0.5": int((d.frac_over_half < 0.5).sum()), "frac_over_half_max": d.frac_over_half.max(),
      "median_participation_median": d["median"].median()})).round(3).to_string())
print(PA[PA.reading == "a_own"].round(3).to_string(index=False))

## fig7f-strongly-variable-neurons

> Neurons with high participation in one response often have low participation in another; every
> preparation contains neurons whose change in participation well exceeds that predicted by a
> noise model.

In [ ]:
rows = []
for (reading, p), (P, ds) in PART.items():
    dP = np.column_stack([P[0] - P[1], P[1] - P[2], P[0] - P[2]])
    mx = np.abs(dP[:, :2]).max(1)
    for pool in ["all_pairs", "consecutive"]:
        data = dP.ravel(order="F") if pool == "all_pairs" else dP[:, :2].ravel(order="F")
        mu, sd, nout = apl.iterate_outliers(data)
        thr = max(abs(mu - 3 * sd), abs(mu + 3 * sd))
        nv = int((mx >= thr).sum())
        rows.append({"reading": reading, "pool": pool, "prep": p, "neurons": P.shape[1], "mu": mu, "sd": sd, "dropped": nout,
                     "threshold": thr, "strongly_variable": nv, "proportion": nv / P.shape[1]})
SV = pd.DataFrame(rows)
print(SV.groupby(["reading", "pool"]).apply(lambda d: pd.Series({"preps_with_one": int((d.strongly_variable > 0).sum()),
      "min_count": d.strongly_variable.min(), "proportion_range": f"{d.proportion.min():.3f} to {d.proportion.max():.3f}"})).to_string())
print(SV.round(4).to_string(index=False))

## halving-participation

> Even the most strongly participating neurons in a response could more than halve their
> participation in other evoked responses.

In [ ]:
rows = []
for (reading, p), (P, ds) in PART.items():
    for sel_name in ["top_neuron", "top_10pct"]:
        hit, examples = False, []
        for a in range(3):
            sel = [int(np.argmax(P[a]))] if sel_name == "top_neuron" else list(np.argsort(P[a])[::-1][:int(np.ceil(0.1 * P.shape[1]))])
            for b in range(3):
                if b == a:
                    continue
                lo = [n for n in sel if P[b][n] < 0.5 * P[a][n]]
                if lo:
                    hit = True
                    examples.append((a + 1, b + 1, len(lo), round(float(min(P[b][n] / P[a][n] for n in lo)), 3)))
        rows.append({"reading": reading, "selection": sel_name, "prep": p, "halves": hit, "examples": examples[:3]})
HV = pd.DataFrame(rows)
print(HV.groupby(["reading", "selection"]).halves.sum().to_string())
print(HV[HV.reading == "a_own"].to_string(index=False))

## fig7s1-rate-synchrony

> A neuron's participation reflects both its firing rate and its synchrony; firing rate explains on
> average 50% of the variation in participation within a response, and rate changes 23% of the
> change in participation between responses.

In [ ]:
rows = []
for reading in ["a_own", "b_common"]:
    for rate_from in [32.5, 30.0]:
        r2, dr2, dr2_code = [], [], []
        for p in range(1, 11):
            ix = sorted([i for i, r in enumerate(recs) if r["prep"] == p], key=lambda i: recs[i]["bout"])
            P = PART[(reading, p)][0]
            rates = np.array([[np.sum(tr >= rate_from) / (progs[i].t_end - rate_from) for tr in progs[i].trains] for i in ix])
            r2 += [np.corrcoef(P[b], rates[b])[0, 1] ** 2 for b in range(3)]
            dR = np.column_stack([rates[0] - rates[1], rates[0] - rates[2], rates[1] - rates[2]])
            dPm = np.column_stack([P[0] - P[1], P[0] - P[2], P[1] - P[2]])
            dPc = np.column_stack([P[0] - P[1], P[1] - P[2], P[0] - P[2]])
            dr2 += [np.corrcoef(dR[:, c], dPm[:, c])[0, 1] ** 2 for c in range(3)]
            dr2_code += [np.corrcoef(dR[:, c], dPc[:, c])[0, 1] ** 2 for c in range(3)]
        rows.append({"reading": reading, "rate_from_s": rate_from, "mean_R2_within": np.mean(r2), "range_within": f"{min(r2):.2f} to {max(r2):.2f}",
                     "mean_R2_change_matched": np.mean(dr2), "mean_R2_change_code_pairing": np.mean(dr2_code), "n_programs": len(r2), "n_pairs": len(dr2)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

## fig7gh-robust-to-participation

> The distance between a pair of responses does not correlate with the total change in neuron
> participation between them nor with the distance between their participation distributions; the
> program is robust to the participation of single neurons.

In [ ]:
rows = []
for k in ["own/min80", "pooled/le80"]:
    for ps in ["recurrent", "code"]:
        T = haus_table(k, "prog1", ps)
        T["norm_dist"] = T.H / T.ctrl_mean
        for reading in ["a_own", "b_common"]:
            tc, hd = [], []
            for row in T.itertuples():
                P = PART[(reading, row.prep)][0]
                i, j = [int(x) - 1 for x in row.pair.split(",")]
                tc.append(np.abs(P[i] - P[j]).sum())
                hd.append(apl.hellinger(P[i], P[j]))
            T["total_change"], T["hellinger"] = tc, hd
            keep = ~((T.prep == 4) & T.pair.isin(["1,2", "1,3"]))
            for x in ["total_change", "hellinger"]:
                X = sm.add_constant(T.loc[keep, x]); y = T.loc[keep, "norm_dist"]
                ols = sm.OLS(y, X).fit()
                rob = sm.RLM(y, X, M=sm.robust.norms.TukeyBiweight()).fit()
                ck = T.norm_dist < 1
                ols_code = sm.OLS(T.loc[ck, "norm_dist"], sm.add_constant(T.loc[ck, x])).fit()
                rows.append({"reading": k, "points": ps, "participation": reading, "x": x, "n": int(keep.sum()),
                             "slope": ols.params.iloc[1], "p": ols.pvalues.iloc[1], "R2": ols.rsquared,
                             "robust_slope": rob.params.iloc[1], "robust_p": rob.pvalues.iloc[1],
                             "code_exclusion_n": int(ck.sum()), "code_exclusion_p": ols_code.pvalues.iloc[1]})
RB = pd.DataFrame(rows)
print(RB.round(4).to_string(index=False))

## fig7s2-no-invariant-core

placeholder

## fig8-caudolateral

> Neurons with strong participation across the three responses, and those with highly variable
> participation, lie mainly in the caudo-lateral quadrant of the ganglion.

Blocked (see the header): neither the data set, its documentation nor the paper's text says which
directions of the array's x and y axes are caudal and lateral. The counts by quadrant of the array
coordinates are shown so that the claim can be judged once the orientation is known; they decide
nothing here.

In [ ]:
rows = []
for side, group in [("left", [1, 2, 4, 10]), ("right", [3, 5, 6, 9])]:
    vals_max, vals_rng, xs, ys = [], [], [], []
    for p in group:
        P = PART[("a_own", p)][0]
        vals_max.append(P.max(0)); vals_rng.append(P.max(0) - P.min(0))
        xs.append(D["coords"][p - 1]["x"]); ys.append(D["coords"][p - 1]["y"])
    allmax = np.concatenate([PART[("a_own", p)][0].max(0) for p in [1, 2, 3, 4, 5, 6, 9, 10]])
    allrng = np.concatenate([np.ptp(PART[("a_own", p)][0], 0) for p in [1, 2, 3, 4, 5, 6, 9, 10]])
    x, y = np.concatenate(xs), np.concatenate(ys)
    cx, cy = 25.0, 25.0     # centre of the 501 x 501 template, in coordinate units
    for name, v, allv in [("max", np.concatenate(vals_max), allmax), ("range", np.concatenate(vals_rng), allrng)]:
        edges = np.linspace(allv.min(), allv.max(), 6)
        top = v >= edges[-2]
        q = np.where(x >= cx, "x+", "x-").astype(object) + np.where(y >= cy, "y+", "y-")
        counts = pd.Series(q[top]).value_counts().to_dict()
        rows.append({"side": side, "measure": name, "neurons": v.size, "top_fifth": int(top.sum()), "by_quadrant": counts})
print(pd.DataFrame(rows).to_string(index=False))

## Summary

placeholder

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (main process)")